In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.retrieval.retriever import Retriever, preprocess_query

retriever = Retriever(
    persist_dir='../vector_store',
    collection_name='sr_papers',
)

info = retriever.collection_info()
print('Collection info:')
for k, v in info.items():
    print(f'  {k}: {v}')

/Users/madhumithakatam/Documents/Projects/SR_RAG/sr-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection info:
  collection_name: sr_papers
  total_chunks: 695
  persist_dir: ../vector_store
  embedding_model: sentence-transformers/all-MiniLM-L6-v2


In [2]:
def show(result):
    print(f'Query   : {result["query"]}')
    print(f'Latency : {result["latency_ms"]:.0f} ms')
    print('-' * 60)
    for r in result['results']:
        print(f"[{r['citation_index']}] {r['method']} | p{r['page_number']} | score={r['score']:.3f}")
        print(f"    {r['text'][:180]}...")
        print()

show(retriever.retrieve('What loss function does SRGAN use?', top_k=4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25781.07it/s]


Query   : What loss function does SRGAN use
Latency : 2747 ms
------------------------------------------------------------
[1] SRGAN / SRResNet | p7 | score=0.692
    Table 1: Performance of different loss functions for SR-
ResNet and the adversarial networks on Set5 and Set14
benchmark data. MOS score significantly higher (p < 0.05)
than with o...

[2] SRGAN / SRResNet | p7 | score=0.650
    V GG/5.4 with φ5,4, a loss defined
on feature maps of higher level features from deeper
network layers with more potential to focus on the
content of the images [68, 65, 40]. We re...

[3] SRGAN / SRResNet | p6 | score=0.617
    lcue6vlrd01ljkdtdkhmfvk7vtjhetog
5https://github.com/david-gpu/srez
convolutional. We scaled the range of the LR input images
to [0, 1] and for the HR images to [−1, 1]. The MSE lo...



In [3]:
show(retriever.retrieve('How does SwinIR use transformer attention?', top_k=4))

Query   : How does SwinIR use transformer attention
Latency : 91 ms
------------------------------------------------------------
[1] HAT | p13 | score=0.643
    SwinIR. 2) We enlarge SwinIR by increasing the width and
depth of SwinIR to achieve similar computations as HAT,
denoted as SwinIR-L1 and SwinIR-L2. HAT achieves the
best performan...

[2] HAT | p1 | score=0.618
    mechanism and utilize long-range information. Thus, we
employ the attribution analysis method LAM [15] to ex-
amine the involved range of utilized information for recon-
struction ...

[3] HAT | p1 | score=0.616
    intermediate features of SwinIR, as depicted in Fig. 3. It
demonstrates that the shift window mechanism cannot per-
fectly realize cross-window information interaction.
1
arXiv:220...

[4] SwinIR | p5 | score=0.601
    Note that RCAN and
HAN introduce channel and spatial attention, IGNN pro-
poses adaptive patch feature aggregation, and NLSA is
based on the non-local attention mechanism. However,...



In [4]:
show(retriever.retrieve('Which papers use DIV2K for training?', top_k=5))

Query   : Which papers use DIV2K for training
Latency : 39 ms
------------------------------------------------------------
[1] EDSR | p7 | score=0.490
    performance and blue indicates the second best. Note that DIV2K validation results are acquired from published demo
codes.
4.5. Benchmark Results
We provide the quantitative evalua...

[2] EDSR | p4 | score=0.438
    4. Experiments
4.1. Datasets
DIV2K dataset [26] is a newly proposed high-quality
(2K resolution) image dataset for image restoration tasks.
The DIV2K dataset consists of 800 traini...

[3] RDN | p5 | score=0.403
    the whole preceding layers in a global way in the LR space.
5. Experimental Results
The source code of the proposed method can be down-
loaded at https://github.com/yulunzhang/RDN....

[4] HAT | p5 | score=0.395
    ing rate for fine-tuning are very important for the effective-
ness of the pre-training strategy. We think that it is because
Transformer requires more data and iterations to learn...

[5] EDSR | 

In [5]:
show(retriever.retrieve_by_method('generator architecture', method='ESRGAN', top_k=3))

Query   : generator architecture
Latency : 75 ms
------------------------------------------------------------
[1] ESRGAN | p9 | score=0.420
    the trained PSNR-oriented model as an initialization for the generator. The
generator is trained using the loss function in Eq. (3) with λ = 5×10−3 and η =
1×10−2. The learning rat...

[2] ESRGAN | p14 | score=0.346
    judge whether one image is more realistic than another, guiding the generator
to recover more detailed textures. Moreover, we have enhanced the perceptual
loss by using the feature...

[3] ESRGAN | p5 | score=0.287
    tion is done in the LR feature space. We could select or design "basic blocks"
(e.g., residual block [18], dense block [34], RRDB) for better performance.
3.1
Network Architecture
...



In [9]:
# Only transformer-era papers (2021+)
show(retriever.retrieve_by_year_range('attention mechanism', 2021, 2023, top_k=4))

ValueError: Expected operator expression to have exactly one operator, got {'$gte': 2021, '$lte': 2023} in query.

In [7]:
test_queries = [
    '  What is PSNR?  ',
    'How does RCAN work???',
    'residual   dense   network',
    'SRGAN vs ESRGAN',
]
for q in test_queries:
    cleaned = preprocess_query(q)
    print(f'  "{q}"  ->  "{cleaned}"')

  "  What is PSNR?  "  ->  "What is PSNR"
  "How does RCAN work???"  ->  "How does RCAN work"
  "residual   dense   network"  ->  "residual dense network"
  "SRGAN vs ESRGAN"  ->  "SRGAN vs ESRGAN"


In [8]:
queries = [
    'perceptual loss function',
    'GAN discriminator training',
    'residual channel attention',
    'bicubic interpolation baseline',
    'PSNR SSIM image quality metrics',
    'real world image degradation',
    'swin transformer window attention',
    'dense connections feature reuse',
    'image upscaling factor 4x',
    'batch normalisation removed EDSR',
]

rows = []
for q in queries:
    result = retriever.retrieve(q, top_k=5)
    scores = [r['score'] for r in result['results']]
    top_methods = [r['method'] for r in result['results'][:3]]
    rows.append({
        'Query':       q[:40],
        'Top-1 score': round(scores[0], 3) if scores else 0,
        'Top-3 avg':   round(sum(scores[:3]) / min(3, len(scores)), 3) if scores else 0,
        'Latency ms':  result['latency_ms'],
        'Top methods': ', '.join(top_methods),
    })

df = pd.DataFrame(rows)
print(f"Average top-1 score: {df['Top-1 score'].mean():.3f}")
print(f"Min top-1 score    : {df['Top-1 score'].min():.3f}")
print()
df

Average top-1 score: 0.607
Min top-1 score    : 0.345



,Query,Top-1 score,Top-3 avg,Latency ms,Top methods
0,perceptual loss function,0.641,0.590,93.5,"ESRGAN, SRGAN / SRResNet, SRGAN / SRResNet"
1,GAN discriminator training,0.698,0.668,69.6,"ESRGAN, ESRGAN, ESRGAN"
2,residual channel attention,0.682,0.675,22.8,"RCAN, RCAN, RCAN"
3,bicubic interpolation baseline,0.534,0.514,10.4,"SRCNN, SRGAN / SRResNet, SRGAN / SRResNet"
4,PSNR SSIM image quality metrics,0.713,0.669,25.6,"HAT, SwinIR, RCAN"
5,real world image degradation,0.712,0.698,33.1,"Real-ESRGAN, Real-ESRGAN, Real-ESRGAN"
6,swin transformer window attention,0.692,0.628,10.2,"SwinIR, HAT, HAT"
7,dense connections feature reuse,0.489,0.482,9.7,"RDN, RDN, RDN"
8,image upscaling factor 4x,0.564,0.536,10.3,"SRGAN / SRResNet, RCAN, Real-ESRGAN"
9,batch normalisation removed EDSR,0.345,0.321,9.7,"ESRGAN, ESRGAN, EDSR"
